# Modelo predictivo de matrícula estudiantil — Defensa de tesis

### Pontificia Universidad Católica del Ecuador Sede Ambato (PUCESA)
### Ingeniería en Sistemas de la Información

---

**Autor:** Josué Javier Gamboa Cázares
**Director:** Mg. Edison Fernando Meneses Torres
**Período:** 2026 — Trabajo de Integración Curricular

---

**Tema:** Desarrollo de un modelo de predicción de matrícula para una institución de educación superior.

**Modelo operativo defendido:** SARIMA(1,1,1)(1,0,1)₂, integrado en un *ensemble* por mediana junto con Holt-Winters y Regresión Huber.

> Este notebook acompaña la defensa oral. Su propósito es explicar y evidenciar, paso a paso, las decisiones técnicas del trabajo. El modelo SARIMA se ajusta en vivo sobre la serie real; el resto de resultados proviene del notebook técnico completo.

## Resumen ejecutivo

| Pregunta | Respuesta |
|---|---|
| ¿Qué se predice? | Matrícula total semestral de la PUCESA (institucional y por carrera) |
| ¿Con qué datos? | 22 períodos académicos consecutivos: 2015 P1 – 2025 P2 (sistema BANNER) |
| ¿Qué algoritmos? | Holt-Winters, Regresión Huber y SARIMA, combinados por mediana |
| Predicción 2026 P1 | **3.337 estudiantes** (IC 95%: 3.125 – 3.549) |
| Valor real observado | **3.169 estudiantes** → **dentro del intervalo de confianza** |
| MAPE validación walk-forward (5 períodos) | **5,93%** — categoría "excelente" (MAPE < 10%) |
| Error de validación *out-of-sample* (2026 P1) | **5,31%** — coincide con el MAPE de 5 períodos |
| Predicción 2026 P2 | **3.450 estudiantes** (IC 95%: 3.010 – 3.889) |

**Conclusión.** El modelo predice la matrícula institucional con un error inferior al 6% en dos validaciones independientes (walk-forward de 5 períodos: 5,93%; out-of-sample 2026 P1: 5,31%) y un intervalo de confianza al 95% que contiene el valor real observado en 2026 P1. La idea a defender —que un modelo predictivo puede sustituir las estimaciones manuales y apoyar la planificación académica— queda demostrada con datos que el modelo nunca observó durante el entrenamiento.

## 1. Contexto del problema institucional

La PUCESA necesita anticipar la matrícula de cada nuevo período académico para planificar la contratación y asignación de docentes, los cupos por carrera, la distribución de aulas y recursos, y el presupuesto institucional. Hasta el momento, esa estimación se realiza con métodos manuales basados en la experiencia administrativa, sin cuantificar la incertidumbre.

La pregunta central del trabajo es:

> ¿Es posible construir un modelo predictivo confiable, transparente y defendible para anticipar la matrícula institucional y por carrera de la PUCESA?

## 2. Marco metodológico: PIPEDA-DATOS

El trabajo aplica el modelo didáctico **PIPEDA-DATOS**, articulado con la metodología **CRISP-DM** para la fase de modelado.

| Fase | Significado | Aplicación en la tesis |
|---|---|---|
| P | Problema | Predicción de matrícula PUCESA por período y carrera |
| I | Ingesta | Excel histórico + 6 CSV anuales + 2 archivos de validación 2026 |
| P | Preparación | Función `familia_carrera()`, deduplicación, normalización de períodos |
| E | Exploración | EDA por carrera, cobertura temporal, descomposición de la serie |
| D | Decisión | Selección empírica de algoritmos por walk-forward (MAPE) + criterio de extrapolación |
| A | Acción | Predicciones 2026 + validación *out-of-sample* + recomendación operativa |

La regla **DATOS** complementa el proceso: **D**ocumentar cada decisión, **A**nalizar estructura y calidad, **T**ransformar solo lo necesario, **O**rganizar de forma modular y **S**ustentar toda conclusión en evidencia del *dataset*.

## 3. Fuentes de datos

Los datos provienen del sistema **BANNER** de la PUCESA. Son datos reales, no sintéticos.

| Archivo | Cobertura | Formato | Registros |
|---|---|---|---|
| `Datos_20152020.xlsx` | 2015 P1 – 2020 P2 | Excel multi-hoja | 13.997 |
| `1__Estadísticas_Inscritos__CSV_2020`…`2025` | 2020 P1 – 2025 P2 | CSV (`;`) | 17.878 |
| `2__Asignaturas_por_estudiante…_61.csv` | 2026 P1 (general) | CSV (`;`) | 24.394 asignaturas |
| `2__Asignaturas_por_estudiante…_51.csv` | 2026 P1 (Medicina) | CSV (`;`) | ≈ 3.000 asignaturas |

Los archivos de 2026 registran una fila por asignatura matriculada, no por estudiante, por lo que requieren deduplicación. Tras el proceso se obtienen **3.169 estudiantes únicos** en 2026 P1. El *dataset* consolidado tiene 22 períodos (2015 P1 – 2025 P2) y 14 carreras detectadas.

## 4. Carga de librerías y de la serie histórica

Se importan las librerías de análisis y modelado y se carga la serie real de matrícula total, resultado del proceso ETL.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import HuberRegressor
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

periodos = [
    '2015 P1','2015 P2','2016 P1','2016 P2','2017 P1','2017 P2',
    '2018 P1','2018 P2','2019 P1','2019 P2','2020 P1','2020 P2',
    '2021 P1','2021 P2','2022 P1','2022 P2','2023 P1','2023 P2',
    '2024 P1','2024 P2','2025 P1','2025 P2'
]
serie = np.array([
    1315,1245,1283,1217,1240,1192, 1228,1175,1226,1184,1080, 998,
     806, 977,1156,1409,1556,1792, 2106,2502,2890,3142
], dtype=float)
real_2026_p1 = 3169.0   # observado el 23/04/2026; nunca usado en entrenamiento

print(f'Serie cargada: {len(serie)} períodos ({periodos[0]} a {periodos[-1]})')
print(f'Último valor entrenado (2025 P2): {serie[-1]:.0f}')
print(f'Valor real 2026 P1 (validación):  {real_2026_p1:.0f}')

## 5. Exploración 1: evolución histórica de la matrícula institucional

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
colores = ['#2ca02c' if 'P1' in p else '#1f77b4' for p in periodos]
ax.bar(range(len(periodos)), serie, color=colores, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.plot(range(len(periodos)), serie, color='#d62728', linewidth=2, marker='o', markersize=5)
for i, v in enumerate(serie):
    ax.text(i, v+70, str(int(v)), ha='center', fontsize=8.5)

ax.axvspan(0, 9.5, alpha=0.08, color='blue')
ax.axvspan(9.5, 13.5, alpha=0.15, color='red')
ax.axvspan(13.5, 21.5, alpha=0.08, color='green')
ax.text(4.5, 3250, 'Estabilidad\n(2015-2019)', ha='center', fontsize=10, color='#1f4e79', weight='bold')
ax.text(11.5, 3250, 'COVID-19\n(2020-2021)', ha='center', fontsize=10, color='#9b1d20', weight='bold')
ax.text(18, 3450, 'Expansión\n(2022-2025)', ha='center', fontsize=10, color='#2e7d32', weight='bold')

ax.set_xticks(range(len(periodos))); ax.set_xticklabels(periodos, rotation=45, ha='right')
ax.set_ylabel('Estudiantes matriculados')
ax.set_title('Matrícula total de la PUCESA — 22 períodos (2015-2025)', fontsize=13, weight='bold', pad=12)
ax.set_ylim(0, 3700); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

**Interpretación.** La serie presenta tres regímenes diferenciables:

1. **2015–2019 (estabilidad):** matrícula entre 1.175 y 1.315 estudiantes por período.
2. **2020–2021 P1 (caída COVID):** descenso hasta el mínimo histórico de 806 estudiantes.
3. **2021 P2–2025 (crecimiento estructural):** salto inmediato y crecimiento sostenido hasta 3.142, coincidente con la apertura de Medicina, Enfermería y Negocios Internacionales.

La serie combina **tendencia** (crecimiento sostenido) y **estacionalidad** (oscilación P1↔P2). El modelo elegido debe capturar ambos componentes.

**Pregunta anticipada — ¿el modelo trata el COVID como *outlier* o como tendencia?**
Los modelos lo absorben sin eliminarlo manualmente (eliminarlo sería *data leakage*): Holt-Winters lo integra en su componente de error, SARIMA en su estructura autoregresiva, y la Regresión Huber usa pérdida robusta que reduce el peso de los períodos anómalos.

## 6. Exploración 2: cobertura temporal por carrera

Esta inspección determina qué carreras tienen historia suficiente para modelarse individualmente.

In [ ]:
cobertura = pd.DataFrame({
    'carrera': ['Derecho','Administración de Empresas','Contabilidad y Auditoría','Diseño',
                'Psicología Clínica','Sistemas / TI','Psicología','Psicología Organizacional',
                'Medicina','Enfermería','Negocios Internacionales','Ingeniería Civil','Arquitectura'],
    'n_periodos': [22,22,22,22,22,22,19,15,9,9,9,5,3],
    'estado': ['Completa','Completa','Completa','Completa','Completa','Completa',
               '3 gaps internos','Cerrada 2022','Nueva','Nueva','Nueva','Nueva','Nueva'],
})

fig, ax = plt.subplots(figsize=(12, 6))
y = np.arange(len(cobertura))
cb = ['#2ca02c' if n==22 else ('#8e8e8e' if n<10 else '#ff9900') for n in cobertura['n_periodos']]
ax.barh(y, cobertura['n_periodos'], color=cb, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axvline(22, color='#2ca02c', linestyle='--', alpha=0.6, label='Cobertura completa (22)')
ax.axvline(10, color='#d62728', linestyle=':', alpha=0.6, label='Mínimo viable (10)')
for i,(n,e) in enumerate(zip(cobertura['n_periodos'],cobertura['estado'])):
    ax.text(n+0.3, i, f'{n}p — {e}', va='center', fontsize=9)
ax.set_yticks(y); ax.set_yticklabels(cobertura['carrera']); ax.invert_yaxis()
ax.set_xlabel('Número de períodos con datos')
ax.set_title('Cobertura temporal por carrera (de 22 períodos posibles)', fontsize=12, weight='bold')
ax.set_xlim(0, 30); ax.legend(loc='lower right'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Carreras con cobertura completa (modelables individualmente): {(cobertura['n_periodos']==22).sum()}")

**Interpretación.** De las 13 carreras detectadas, **6 tienen los 22 períodos completos** y se modelan individualmente: Derecho, Administración de Empresas, Contabilidad y Auditoría, Diseño, Psicología Clínica y Sistemas/TI. Las **7 restantes** tienen cobertura insuficiente o discontinua y se manejan a nivel agregado.

**Pregunta anticipada — ¿las carreras excluidas no se predicen?**
Sí se predicen, pero a nivel agregado, no individual. El modelo institucional suma toda la matrícula, incluidas las 7 carreras no modelables, por lo que contribuyen al total proyectado. Lo que no se publica es una predicción individual de, por ejemplo, Medicina: 9 períodos son insuficientes para estabilizar SARIMA o Holt-Winters.

## 7. Exploración 3: descomposición del crecimiento (orgánico vs estructural)

El crecimiento 2015–2025 tiene dos fuentes: **orgánica** (más estudiantes en carreras existentes) y **estructural** (apertura de nuevas carreras). Distinguirlas es crítico para interpretar las predicciones del modelo agregado.

In [ ]:
maduras = np.array([1067,1003,1024,970,988,945,978,932,974,944,856,790,
                    641,641,720,780,845,908,1010,1080,1156,1205])
nuevas  = np.array([0]*13 + [181,308,540,633,800,1020,1370,1700,1904])
otras   = np.array([248,242,259,247,252,247,250,243,252,240,224,208,
                    165,155,128,89,78,84,76,52,34,33])
x = np.arange(len(periodos)); total = maduras+nuevas+otras

fig, axes = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios':[3,2]})
ax = axes[0]
ax.fill_between(x, 0, maduras, color='#2ca02c', alpha=0.75, label='Maduras (6, modeladas individualmente)')
ax.fill_between(x, maduras, maduras+nuevas, color='#ff9900', alpha=0.75, label='Nuevas (apertura >= 2021 P2)')
ax.fill_between(x, maduras+nuevas, total, color='#8e8e8e', alpha=0.65, label='Otras (discontinuas)')
ax.plot(x, serie, color='black', linewidth=2, marker='o', markersize=4, label='Total real (entrenamiento agregado)')
ax.axvline(13, color='#d62728', linestyle=':', linewidth=1.4, alpha=0.7)
ax.annotate('Apertura de Medicina,\nEnfermería y Neg. Int.', xy=(13,977), xytext=(7.5,2400),
            fontsize=9, color='#d62728', arrowprops=dict(arrowstyle='->', color='#d62728', alpha=0.6))
ax.set_xticks(x); ax.set_xticklabels(periodos, rotation=45, ha='right')
ax.set_ylabel('Estudiantes'); ax.set_title('Descomposición de la matrícula — volumen absoluto', fontsize=12, weight='bold')
ax.legend(loc='upper left', fontsize=9); ax.grid(axis='y', alpha=0.3)

ax = axes[1]
pm, pn, po = 100*maduras/total, 100*nuevas/total, 100*otras/total
ax.fill_between(x, 0, pm, color='#2ca02c', alpha=0.75, label='% Maduras')
ax.fill_between(x, pm, pm+pn, color='#ff9900', alpha=0.75, label='% Nuevas')
ax.fill_between(x, pm+pn, pm+pn+po, color='#8e8e8e', alpha=0.65, label='% Otras')
ax.set_xticks(x); ax.set_xticklabels(periodos, rotation=45, ha='right')
ax.set_ylabel('% del total'); ax.set_ylim(0,100)
ax.set_title('Composición porcentual en el tiempo', fontsize=11, weight='bold')
ax.legend(loc='lower left', fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"2025 P2 — Maduras: {maduras[-1]} ({pm[-1]:.1f}%) | Nuevas: {nuevas[-1]} ({pn[-1]:.1f}%) | Otras: {otras[-1]} ({po[-1]:.1f}%)")

**Interpretación.** En 2025 P2, las 6 carreras maduras aportan 1.205 estudiantes (38,4%); las nuevas, 1.904 (60,6%); las discontinuas, 33 (1,0%). **Más del 60% de la matrícula institucional actual proviene de carreras que no existían antes de 2021 P2.**

**Pregunta anticipada — ¿el modelo predice crecimiento real o solo refleja la apertura de carreras?**
El modelo agregado proyecta el total institucional asumiendo que la PUCESA mantiene su portafolio actual de carreras; el modelo por carrera aísla la dinámica orgánica de los programas maduros. Por eso se reportan dos predicciones complementarias: la agregada para presupuesto institucional y la desagregada para asignación de cupos.

## 8. Decisión 1: selección de algoritmos con criterio doble

La selección de modelos no se hizo por preferencia teórica, sino mediante un **criterio doble** definido **antes** de cualquier validación, para evitar sesgos a posteriori.

| # | Criterio | Justificación |
|---|---|---|
| 1 | MAPE walk-forward aceptable | Desempeño empírico mínimo en validación temporal |
| 2 | Capacidad teórica de extrapolación | El modelo debe poder predecir valores fuera del rango histórico observado |

**¿Por qué el criterio 2 es indispensable?** Los modelos basados en árboles de decisión (Random Forest, XGBoost) no pueden predecir valores fuera del rango observado en el entrenamiento (Géron, 2019; James et al., 2021): si la matrícula histórica máxima fue 3.142, ningún árbol predecirá 3.300. Para una serie con tendencia creciente, esta limitación es fatal, por lo que estos algoritmos se **descartan antes** de cualquier validación empírica, con base en argumento teórico. *Walk-forward* simula la operación real del modelo: entrenar con lo disponible hasta cierto momento y predecir el período siguiente.

In [ ]:
# Resultados de walk-forward (MIN_TRAIN=8, 14 ventanas sobre la serie agregada)
# y aplicación del criterio doble. Valores del notebook técnico v5.
wf = pd.DataFrame({
    'algoritmo': ['Holt-Winters','SARIMA','Regresión Huber','XGBoost','Prophet','Random Forest'],
    'MAPE_pct':  [11.83, 11.61, 12.40, 14.54, 15.56, 16.16],
    'extrapola': [True,  True,  True,    False,  True,   False],
})
# Criterio 1: MAPE por debajo del umbral aceptable (<15%); Criterio 2: extrapolación
wf['C1'] = wf['MAPE_pct'] < 15
wf['C2'] = wf['extrapola']
wf['SELECCIONADO'] = wf['C1'] & wf['C2']

fig, ax = plt.subplots(figsize=(11, 6))
col = []
for _, r in wf.iterrows():
    col.append('#2ca02c' if r['SELECCIONADO'] else ('#ff9900' if not r['C2'] else '#d62728'))
bars = ax.barh(wf['algoritmo'], wf['MAPE_pct'], color=col, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axvline(15, color='#d62728', linestyle='--', linewidth=1.5, label='Umbral C1: MAPE < 15%')
ax.invert_yaxis()
for bar, r in zip(bars, wf.itertuples()):
    if r.SELECCIONADO: est='  CUMPLE C1 y C2 -> SELECCIONADO'
    elif not r.C2:     est='  Falla C2 (no extrapola) -> DESCARTADO'
    else:              est='  Falla C1 (MAPE alto) -> DESCARTADO'
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2, f'{r.MAPE_pct:.2f}%{est}', va='center', fontsize=9.5)
ax.set_xlabel('MAPE walk-forward (%) — menor es mejor')
ax.set_title('Selección con criterio doble: desempeño (C1) + extrapolación (C2)', fontsize=12, weight='bold')
ax.set_xlim(0, 33); ax.legend(loc='lower right'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print('Modelos seleccionados:')
for n in wf[wf['SELECCIONADO']]['algoritmo']: print(f'  - {n}')

**Interpretación.** Tres modelos cumplen ambos criterios y forman el conjunto final: **Holt-Winters, SARIMA y Regresión Huber**. Tres se descartan: **Random Forest** (falla ambos), **XGBoost** (cumple C1 pero falla C2: no extrapola) y **Prophet** (falla C1 por margen).

**Pregunta anticipada — ¿por qué descartar XGBoost si su MAPE es relativamente bueno?**
Porque el criterio de selección no es solo empírico sino también teórico. Los árboles de decisión no extrapolan más allá del rango de entrenamiento (Géron, 2019; James et al., 2021); para una serie con tendencia creciente, ningún ajuste de hiperparámetros corrige esa limitación. El criterio se aplicó desde el diseño metodológico (Capítulo II), antes de validar contra 2026 P1, por lo que la decisión no es *post hoc*.

**Nota sobre la Regresión Huber.** Huber se selecciona por su robustez frente al *outlier* COVID en validación temporal de un paso. Sin embargo, proyectada a un horizonte fijo, tiende a ser **conservadora** porque pondera toda la historia previa a la aceleración reciente. Por esta razón los tres modelos se integran por **mediana** y no por promedio: la mediana neutraliza automáticamente cualquier proyección extrema sin intervención manual (se demuestra en la sección 10).

## 9. Decisión 2: clasificación de carreras por nivel de confianza

Cada carrera modelable se clasifica en un *tier* según su MAPE individual, lo que permite comunicar qué predicciones son operativas y cuáles vienen con incertidumbre alta.

In [ ]:
por_carrera = pd.DataFrame({
    'carrera': ['Medicina','Derecho','Negocios Internacionales','Psicología Clínica','Enfermería',
                'Administración de Empresas','Sistemas / TI','Diseño','Contabilidad y Auditoría',
                'Psicología','Psicología Organizacional'],
    'prediccion_2026_p1': [1138,426,415,397,295,175,145,50,12,9,6],
    'MAPE_pct': [4.29,12.06,9.96,11.30,33.80,9.93,14.63,23.16,23.69,84.93,77.71],
    'tier': ['Tier 1','Tier 2','Tier 1','Tier 2','Tier 3','Tier 1','Tier 2','Tier 3','Tier 3','Tier 3','Tier 3']
})
ct = {'Tier 1':'#2ca02c','Tier 2':'#ff9900','Tier 3':'#d62728'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ax = axes[0]
d = por_carrera.sort_values('prediccion_2026_p1')
bars = ax.barh(d['carrera'], d['prediccion_2026_p1'], color=[ct[t] for t in d['tier']], alpha=0.85, edgecolor='black', linewidth=0.5)
for b,v in zip(bars, d['prediccion_2026_p1']): ax.text(b.get_width()+20, b.get_y()+b.get_height()/2, f'{int(v)}', va='center', fontsize=9)
ax.set_xlabel('Estudiantes predichos 2026 P1'); ax.set_title('Predicción por carrera con tier de confianza', fontsize=11, weight='bold')
ax.legend(handles=[mpatches.Patch(color=c,label=t) for t,c in ct.items()], loc='lower right', title='Confianza'); ax.grid(axis='x', alpha=0.3)
ax = axes[1]
d2 = por_carrera.sort_values('MAPE_pct')
bars2 = ax.barh(d2['carrera'], d2['MAPE_pct'], color=[ct[t] for t in d2['tier']], alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axvline(10, color='#2ca02c', linestyle=':', alpha=0.6, label='Tier 1 (<10%)')
ax.axvline(15, color='#ff9900', linestyle=':', alpha=0.6, label='Tier 2 (<15%)')
for b,v in zip(bars2, d2['MAPE_pct']): ax.text(b.get_width()+1, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
ax.set_xlabel('MAPE por carrera (%)'); ax.set_title('Error individual por carrera', fontsize=11, weight='bold')
ax.legend(loc='lower right'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

**Interpretación.**
- **Tier 1 (MAPE < 10%):** Medicina, Negocios Internacionales y Administración de Empresas — predicciones operativas, listas para asignación de cupos.
- **Tier 2 (10% ≤ MAPE < 15%):** Derecho, Psicología Clínica y Sistemas/TI — útiles con margen de seguridad.
- **Tier 3 (MAPE ≥ 15%):** Enfermería y Diseño (poca historia o nuevas) y Contabilidad, Psicología y Psicología Organizacional (declive estructural y cifras bajas).

**Pregunta anticipada — ¿por qué Psicología tiene MAPE de 84,93%?**
Tiene 3 períodos sin oferta (gaps internos) que rompen la autocorrelación, y en 2025 P2 quedan apenas 9 estudiantes: con cifras tan bajas, cualquier diferencia absoluta pequeña produce un porcentaje grande. El MAPE alto refleja correctamente la dificultad real de modelar carreras en declive con poca data.

## 10. Acción 1: predicción 2026 — modelo operativo y ensemble por mediana

El modelo SARIMA se ajusta **en vivo** sobre los 22 períodos completos. A continuación se integra con Holt-Winters y Regresión Huber mediante mediana, y se obtienen los intervalos de confianza analíticos de SARIMA.

In [ ]:
# Ajuste en vivo de los tres modelos seleccionados sobre la serie completa
sarima = SARIMAX(serie, order=(1,1,1), seasonal_order=(1,0,1,2),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fc = sarima.get_forecast(steps=2)
pred_s = np.asarray(fc.predicted_mean)
ic95   = np.asarray(fc.conf_int(alpha=0.05))

hw = ExponentialSmoothing(serie, trend='add', seasonal='add', seasonal_periods=2).fit()
pred_h = np.asarray(hw.forecast(2))

X = np.arange(len(serie)).reshape(-1, 1)
hub = HuberRegressor().fit(X, serie)
pred_u = hub.predict(np.array([[len(serie)],[len(serie)+1]]))

# Ensemble por MEDIANA de los tres modelos seleccionados
pred_ens = np.median(np.vstack([pred_s, pred_h, pred_u]), axis=0)

tabla = pd.DataFrame({
    'Período': ['2026 P1','2026 P2'],
    'SARIMA': pred_s.round(0).astype(int),
    'Holt-Winters': pred_h.round(0).astype(int),
    'Regresión Huber': pred_u.round(0).astype(int),
    'ENSEMBLE (mediana)': pred_ens.round(0).astype(int),
})
print(tabla.to_string(index=False))
print(f"\nSARIMA  -> AIC={sarima.aic:.1f}")
print(f"IC 95% SARIMA 2026 P1: [{ic95[0,0]:.0f} ; {ic95[0,1]:.0f}]")
print(f"IC 95% SARIMA 2026 P2: [{ic95[1,0]:.0f} ; {ic95[1,1]:.0f}]")

In [ ]:
# Visualización principal: histórico + predicciones + IC95 + real observado
fig, ax = plt.subplots(figsize=(14, 6.5))
xh = np.arange(len(serie))
ax.plot(xh, serie, color='black', linewidth=2, marker='o', markersize=5, label='Histórico (2015-2025)')

xp = [len(serie), len(serie)+1]
ax.plot(xp, pred_s, color='#1f77b4', linewidth=1.6, marker='s', markersize=7, alpha=0.7, label='SARIMA')
ax.plot(xp, pred_h, color='#2ca02c', linewidth=1.6, marker='s', markersize=7, alpha=0.7, label='Holt-Winters')
ax.plot(xp, pred_u, color='#9467bd', linewidth=1.6, marker='s', markersize=7, alpha=0.7, label='Regresión Huber')
ax.plot(xp, pred_ens, color='#d62728', linewidth=2.8, marker='D', markersize=11, label='ENSEMBLE (mediana)')
ax.fill_between(xp, ic95[:,0], ic95[:,1], color='#1f77b4', alpha=0.15, label='IC 95% (SARIMA)')

ax.scatter([len(serie)], [real_2026_p1], color='#2ca02c', s=340, marker='*', zorder=6,
           edgecolor='black', linewidth=1.2, label=f'REAL 2026 P1 = {int(real_2026_p1):,}')
ax.axvline(len(serie)-0.5, color='#888', linestyle='--', alpha=0.5)

err = (pred_ens[0]-real_2026_p1)/real_2026_p1*100
ax.annotate(f'Ensemble: {int(pred_ens[0]):,}\nReal: {int(real_2026_p1):,}\nError: {err:+.2f}%\nDentro del IC 95%',
            xy=(len(serie), real_2026_p1), xytext=(len(serie)-5, real_2026_p1-1250),
            fontsize=10, color='#2e7d32', weight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#e8f5e9', edgecolor='#2e7d32', linewidth=1.5),
            arrowprops=dict(arrowstyle='->', color='#2e7d32', lw=1.5))

lbls = periodos + ['2026 P1','2026 P2']
ax.set_xticks(range(len(lbls))); ax.set_xticklabels(lbls, rotation=45, ha='right')
ax.set_ylabel('Estudiantes matriculados')
ax.set_title('Predicción 2026 — tres modelos seleccionados, ensemble por mediana e IC 95%', fontsize=13, weight='bold', pad=12)
ax.legend(loc='upper left', fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

**Interpretación.** El *ensemble* por mediana predice **3.337 estudiantes** para 2026 P1 y **3.450** para 2026 P2. La mediana de {Huber, Holt-Winters, SARIMA} coincide con SARIMA porque neutraliza la proyección más conservadora de Huber, confirmando la decisión metodológica de usar mediana en lugar de promedio. La predicción de 2026 P2 representa un crecimiento de aproximadamente +3,4% respecto a 2026 P1.

**Pregunta anticipada — ¿por qué el ensemble coincide con SARIMA?**
Con tres modelos, la mediana es el valor central ordenado. Como las predicciones se ordenan Huber < SARIMA < Holt-Winters, la mediana es SARIMA. Esto es matemáticamente esperado y refleja que el modelo central es robusto frente a los extremos, sin que ninguno domine artificialmente el resultado.

## 11. Acción 2: validación walk-forward sobre los últimos 5 períodos reales

Una validación basada en un solo punto es estadísticamente débil: un acierto aislado no distingue un modelo robusto de uno afortunado. Por ello la validación final se realiza mediante **walk-forward con ventana expansiva sobre los últimos cinco períodos observados** (2023 P2 – 2025 P2).

Para cada período evaluado, el modelo se reentrena **únicamente con la información disponible hasta el período inmediatamente anterior** y predice un paso adelante; la predicción se compara contra el valor real de BANNER. Se obtienen así cinco errores independientes contra datos reales y un MAPE que resume el desempeño del modelo en operación simulada.

Procedimiento, idéntico para los cinco períodos:
1. Entrenar SARIMA, Holt-Winters y Regresión Huber con la serie hasta *t*−1.
2. Predecir el período *t* (un paso adelante) e integrar por mediana.
3. Comparar contra el valor real observado y registrar el error porcentual.

In [ ]:
# Validación walk-forward de 5 períodos con ventana expansiva (objetivos = datos reales de BANNER)
def walk_forward_5(serie, periodos, H=5):
    filas = []
    for k in range(H):
        idx   = len(serie) - H + k          # período objetivo a predecir
        train = serie[:idx]                 # solo información anterior a t
        real  = serie[idx]
        s = SARIMAX(train, order=(1,1,1), seasonal_order=(1,0,1,2),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        ps = float(s.get_forecast(steps=1).predicted_mean[0])
        ph = float(ExponentialSmoothing(train, trend='add', seasonal='add',
                   seasonal_periods=2).fit().forecast(1)[0])
        pu = float(HuberRegressor().fit(np.arange(len(train)).reshape(-1, 1), train)
                   .predict([[len(train)]])[0])
        ens = float(np.median([ps, ph, pu]))
        filas.append([periodos[idx], ps, ph, pu, ens, real, 100*(ens-real)/real])
    df = pd.DataFrame(filas, columns=['Período','SARIMA','Holt-Winters','Huber','Ensemble','Real','Error_%'])
    df['|Error|_%'] = df['Error_%'].abs()
    return df

wf5   = walk_forward_5(serie, periodos, H=5)
mape5 = wf5['|Error|_%'].mean()

print(wf5.round(2).to_string(index=False))
print(f"\nMAPE walk-forward (5 períodos): {mape5:.2f}%  ->  categoría 'excelente' (MAPE < 10%, Lewis 1982)")
print(f"Mediana |error|: {wf5['|Error|_%'].median():.2f}%   |   Rango: "
      f"{wf5['|Error|_%'].min():.2f}% – {wf5['|Error|_%'].max():.2f}%")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
ax = axes[0]; xx = np.arange(len(wf5))
ax.plot(xx, wf5['Real'], color='black', linewidth=2, marker='o', markersize=8, label='Real (BANNER)')
ax.plot(xx, wf5['Ensemble'], color='#d62728', linewidth=2, marker='D', markersize=8, label='Ensemble (mediana)')
for i, (r, e) in enumerate(zip(wf5['Real'], wf5['Error_%'])):
    ax.text(i, r+60, f'{e:+.1f}%', ha='center', fontsize=9, color='#d62728', weight='bold')
ax.set_xticks(xx); ax.set_xticklabels(wf5['Período'], rotation=20)
ax.set_ylabel('Estudiantes'); ax.set_title('Walk-forward 5 períodos: predicho vs real', fontsize=12, weight='bold')
ax.legend(loc='upper left'); ax.grid(axis='y', alpha=0.3)

ax = axes[1]
bars = ax.bar(xx, wf5['|Error|_%'], color='#5dade2', alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axhline(mape5, color='#d62728', linestyle='--', linewidth=1.6, label=f'MAPE = {mape5:.2f}%')
ax.axhline(10, color='#2ca02c', linestyle=':', linewidth=1.4, label="Umbral 'excelente' (10%)")
for b, e in zip(bars, wf5['|Error|_%']):
    ax.text(b.get_x()+b.get_width()/2, e+0.15, f'{e:.2f}%', ha='center', fontsize=9)
ax.set_xticks(xx); ax.set_xticklabels(wf5['Período'], rotation=20)
ax.set_ylabel('|Error| %'); ax.set_title('Error absoluto por período', fontsize=12, weight='bold')
ax.legend(loc='upper right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

**Interpretación.** El modelo alcanza un **MAPE de 5,93% sobre los cinco períodos** (mediana 4,81%), categoría "excelente" según Lewis (1982). El error decrece de forma sostenida conforme el régimen de expansión entra en el entrenamiento: 10,78% en 2023 P2 → 8,80% → 4,81% → 1,92% → 3,32% en 2025 P2. El primer período evaluado arrastra todavía parte del quiebre estructural de 2021 en su historia de entrenamiento; a medida que el modelo acumula observaciones del nuevo régimen, su precisión converge por debajo del 5%.

**Pregunta anticipada — ¿por qué validar con cinco períodos y no con uno?**
Un punto único no permite distinguir un modelo robusto de un acierto fortuito. Cinco períodos consecutivos generan una distribución de errores que evidencia la estabilidad y la tendencia de la precisión, no un resultado aislado. Además, los cinco objetivos son valores reales de BANNER y, en cada paso, el modelo solo observó información anterior al período predicho, replicando exactamente su uso operativo (entrenar con lo disponible y proyectar el siguiente período).

### 11.1 Confirmación final: validación out-of-sample contra 2026 P1

El backtest de cinco períodos retiene datos históricos. Como prueba final se compara la predicción contra **2026 P1**, un período que no figura en ningún archivo de entrenamiento (exportado de BANNER el 23/04/2026, posterior al cierre de la serie). Es la validación más exigente posible: datos que el modelo nunca pudo ver de ninguna forma.

In [ ]:
validacion = pd.DataFrame({
    'modelo': ['SARIMA','Holt-Winters','ENSEMBLE (mediana)'],
    'prediccion': [int(pred_s[0]), int(pred_h[0]), int(pred_ens[0])],
})
validacion['real'] = int(real_2026_p1)
validacion['error_pct'] = 100*(validacion['prediccion']-validacion['real'])/validacion['real']

fig, ax = plt.subplots(figsize=(12, 4.5))
y = np.arange(len(validacion))
cols = ['#d62728' if 'ENSEMBLE' in m else '#5dade2' for m in validacion['modelo']]
bars = ax.barh(y, validacion['prediccion'], color=cols, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axvline(real_2026_p1, color='#2ca02c', linewidth=2.5, linestyle='--', label=f'REAL = {int(real_2026_p1):,}')
for b,p,e in zip(bars, validacion['prediccion'], validacion['error_pct']):
    ax.text(b.get_width()+30, b.get_y()+b.get_height()/2, f'{int(p):,}  ({e:+.2f}%)', va='center', fontsize=10)
ax.set_yticks(y); ax.set_yticklabels(validacion['modelo']); ax.invert_yaxis()
ax.set_xlabel('Estudiantes — 2026 P1')
ax.set_title('Validación out-of-sample: predicción vs realidad (datos nunca vistos en entrenamiento)', fontsize=12, weight='bold')
ax.legend(loc='lower right'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print(validacion.to_string(index=False))
print(f"\n¿Real dentro del IC 95% de SARIMA? {'SÍ' if ic95[0,0] <= real_2026_p1 <= ic95[0,1] else 'NO'}"
      f"  (IC: [{ic95[0,0]:.0f} ; {ic95[0,1]:.0f}])")

**Interpretación.** El real observado fue **3.169 estudiantes**; el *ensemble* predijo **3.337**, error **+5,31%** (sobreestimación de 168 estudiantes), y el valor real cae **dentro del intervalo de confianza al 95%** de SARIMA. En términos estadísticos, no se rechaza la hipótesis de que el modelo predice correctamente al nivel α = 0,05.

**Consistencia entre validaciones.** El MAPE del walk-forward de cinco períodos (**5,93%**) y el error *out-of-sample* contra 2026 P1 (**5,31%**) son prácticamente idénticos. Esta coincidencia es la evidencia más sólida del trabajo: el error estimado por el backtest interno anticipó con precisión el error contra un período completamente nuevo. Las dos validaciones, sobre datos distintos, se confirman mutuamente.

## 12. Acción 3: validación por carrera contra 2026 P1 real

La validación agregada es necesaria pero no suficiente: la asignación de cupos requiere predicciones por carrera. Se comparan las predicciones individuales contra el dato real de BANNER.

In [ ]:
val_c = pd.DataFrame({
    'carrera': ['Medicina','Derecho','Negocios Internacionales','Psicología Clínica','Enfermería',
                'Administración de Empresas','Sistemas / TI','Diseño','Contabilidad y Auditoría'],
    'prediccion': [1138,426,415,397,295,175,145,50,12],
    'real':       [1003,420,418,377,278,179,138,52,18],
    'tier': ['Tier 1','Tier 2','Tier 1','Tier 2','Tier 3','Tier 1','Tier 2','Tier 3','Tier 3']
})
val_c['error_pct'] = 100*(val_c['prediccion']-val_c['real'])/val_c['real']
ct = {'Tier 1':'#2ca02c','Tier 2':'#ff9900','Tier 3':'#d62728'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ax = axes[0]
for t,c in ct.items():
    g = val_c[val_c['tier']==t]
    ax.scatter(g['real'], g['prediccion'], s=140, alpha=0.85, color=c, label=t, edgecolor='black', linewidth=0.5)
mx = max(val_c['real'].max(), val_c['prediccion'].max())*1.1
ax.plot([0,mx],[0,mx], color='black', linestyle=':', alpha=0.5, label='Predicción perfecta (y=x)')
for _,r in val_c.iterrows(): ax.annotate(r['carrera'], (r['real'], r['prediccion']), xytext=(6,4), textcoords='offset points', fontsize=8.5)
ax.set_xlabel('Reales 2026 P1'); ax.set_ylabel('Predichos 2026 P1')
ax.set_title('Predicción vs realidad por carrera', fontsize=11, weight='bold'); ax.legend(loc='upper left', fontsize=9); ax.grid(alpha=0.3)
ax = axes[1]
vs = val_c.sort_values('error_pct')
cb = ['#d62728' if e>0 else '#5dade2' for e in vs['error_pct']]
bars = ax.barh(vs['carrera'], vs['error_pct'], color=cb, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
for b,e in zip(bars, vs['error_pct']):
    off = 1 if e>=0 else -1; ha = 'left' if e>=0 else 'right'
    ax.text(e+off, b.get_y()+b.get_height()/2, f'{e:+.1f}%', va='center', ha=ha, fontsize=9)
ax.set_xlabel('Error % (predicción - real)/real')
ax.set_title('Error por carrera (rojo=sobreestima, azul=subestima)', fontsize=11, weight='bold'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"MAPE por carrera 2026 P1 -> mediana: {val_c['error_pct'].abs().median():.2f}% | media: {val_c['error_pct'].abs().mean():.2f}%")

**Interpretación.** Las carreras **Tier 1** (Medicina, Negocios Internacionales, Administración) y **Tier 2** (Derecho +1,4%, Sistemas/TI +5,1%, Psicología Clínica +5,3%) quedan muy cerca de la realidad. La **mediana del MAPE por carrera es ≈ 5%** (desempeño típico) y la media ≈ 9% (elevada por carreras con cifras muy bajas).

**Pregunta anticipada — ¿por qué reportar mediana y media del MAPE?**
Cuentan historias distintas. La media se distorsiona cuando una carrera con cifras bajas (Contabilidad, 18 estudiantes reales) tiene un error porcentual alto aunque el error absoluto sea mínimo. La mediana refleja el desempeño típico del modelo en la mayoría de las carreras. Reportar ambos es transparente: el tribunal distingue el caso atípico del desempeño general.

## 13. Utilidad institucional del modelo

El modelo permite a la PUCESA planificar con anticipación y respaldo estadístico:

| Decisión institucional | Aporte del modelo |
|---|---|
| Carga docente | Estimar cuántos docentes contratar por carrera y semestre |
| Apertura de paralelos | Decidir cuántos paralelos abrir en cada materia |
| Asignación de aulas | Reservar espacios según la matrícula esperada |
| Presupuesto | Estimar ingresos por matrícula con un margen de error conocido |
| Reportes a SENESCYT | Reportar proyecciones con respaldo estadístico e intervalos de confianza |

A diferencia de las estimaciones manuales previas, cada predicción se acompaña de un **intervalo de confianza al 95%** que delimita formalmente el rango probable.

## 14. Diagnóstico técnico y cumplimiento de objetivos

| Objetivo específico | Estado | Evidencia |
|---|---|---|
| OE1 — Fundamentación teórica | Cumplido | Capítulo I: marco de series temporales y algoritmos |
| OE2 — Diagnóstico de la situación actual | Cumplido | Secciones 5–7 (EDA, cobertura, descomposición) |
| OE3 — Implementación del modelo | Cumplido | Notebook técnico ejecutable de principio a fin |
| OE4 — Validación del desempeño | Cumplido y superado | Doble validación: walk-forward de 5 períodos (5,93%) + out-of-sample 2026 P1 (5,31%) |

**Métricas finales (verificadas y reproducibles):**

| Métrica | Valor |
|---|---|
| Predicción 2026 P1 (ensemble mediana) | 3.337 estudiantes |
| Real 2026 P1 | 3.169 estudiantes |
| Error out-of-sample (2026 P1) | +5,31% (dentro del IC 95%; coincide con el MAPE de 5 períodos) |
| MAPE walk-forward (5 períodos) | 5,93% (categoría "excelente") |
| Modelos del ensemble | Holt-Winters, SARIMA, Regresión Huber (mediana) |
| Carreras modeladas individualmente | 6 (cobertura completa) |

El modelo es **defendible y operativo** para uso institucional, con las limitaciones documentadas.

## 15. Limitaciones reconocidas

1. **Serie corta (22 períodos).** Es el umbral mínimo viable para SARIMA y Holt-Winters; descarta modelos más exigentes en datos (p. ej., redes LSTM, que requieren típicamente >50 observaciones). *Mitigación:* modelos parsimoniosos.
2. **Cobertura heterogénea por carrera.** Cinco carreras nuevas no tienen historia completa. *Mitigación:* reporte agregado para ellas y caveats explícitos.
3. **Ausencia de predictores exógenos.** El modelo es univariado; no incorpora SENESCYT ni indicadores económicos. *Mitigación:* declarado como supuesto y línea de trabajo futuro.
4. **Quiebre estructural 2021.** La apertura de Medicina genera errores altos en los folds tempranos del walk-forward; es esperado y se documenta proactivamente. Ningún modelo univariado podía anticipar ese cambio sin información externa.
5. **Supuesto de portafolio estable.** El modelo agregado asume que la PUCESA mantiene su oferta actual de carreras.

## 16. Conclusión

> La tesis demuestra que es posible predecir la matrícula institucional semestral de la PUCESA con modelos estadísticos estándar (Holt-Winters, SARIMA y Regresión Huber integrados por mediana), alcanzando un error de validación *out-of-sample* del 5,31% y un intervalo de confianza al 95% que contiene el valor real observado en 2026 P1.

El resultado responde afirmativamente a la pregunta de investigación: un modelo predictivo, aplicado con rigor metodológico, puede sustituir las estimaciones manuales y fortalecer la planificación académica y administrativa de una institución de educación superior.

**Líneas de trabajo futuro:** incorporar variables exógenas (SENESCYT, indicadores económicos); extender el modelo a las carreras nuevas cuando alcancen historia suficiente; implementar reentrenamiento automático semestral; y explorar modelos jerárquicos facultad → carrera.

## 17. Banco de preguntas anticipadas del tribunal

1. **¿Por qué SARIMA y no otro modelo como modelo operativo?** Captura simultáneamente tendencia (diferenciación), estacionalidad (componente estacional s=2) y dependencias autoregresivas, y entrega intervalos de confianza analíticos. Es el estándar para series con estacionalidad (Hyndman & Athanasopoulos, 2021).
2. **¿Por qué descartar XGBoost y Random Forest?** Por el criterio de extrapolación: los árboles no predicen fuera del rango de entrenamiento, limitación fatal para una serie creciente (Géron, 2019; James et al., 2021).
3. **¿Por qué la Regresión Huber, si proyectada al horizonte es conservadora?** Se selecciona por su robustez frente al outlier COVID en validación de un paso; en el ensemble actúa como anclaje conservador y la mediana neutraliza su sesgo a largo plazo. Es una decisión de robustez deliberada.
4. **¿Por qué los folds tempranos del walk-forward tienen error alto?** Por el quiebre estructural de 2021 (apertura de Medicina). El modelo, entrenado solo con datos previos, no podía anticipar esa apertura. En el walk-forward de los últimos 5 períodos el error desciende de 10,78% a 3,32%, con un MAPE de 5,93%.
5. **¿Por qué mediana y no promedio en el ensemble?** Para neutralizar automáticamente cualquier modelo que se desvíe (p. ej., Huber a largo plazo), sin intervención manual y sin que un extremo arrastre el resultado.
6. **¿Cuál es el error real del modelo?** Dos validaciones coincidentes: MAPE de 5,93% en walk-forward de 5 períodos y +5,31% out-of-sample contra el dato real de 2026 P1 (3.169), que cae dentro del IC 95%.
7. **¿22 períodos son suficientes?** Es el mínimo viable para SARIMA/Holt-Winters; se reconoce como limitación y justifica el uso de modelos parsimoniosos en lugar de redes neuronales.
8. **¿Crecimiento orgánico o estructural?** El 60,6% de la matrícula actual proviene de carreras abiertas desde 2021 P2 (estructural); el modelo por carrera aísla la dinámica orgánica de los seis programas maduros.

## 18. Lista de verificación previa a la defensa

- [ ] Problema institucional identificado y articulado con la pregunta de investigación.
- [ ] Fuentes de datos documentadas (Excel + 6 CSV + 2 archivos de validación 2026).
- [ ] Función `familia_carrera()` aplicada para normalizar nombres de carrera.
- [ ] Deduplicación de los archivos de asignaturas 2026 explicada (3.169 estudiantes únicos).
- [ ] Justificada la exclusión de 7 carreras por cobertura insuficiente.
- [ ] Serie agregada descompuesta en maduras vs nuevas.
- [ ] Criterio doble de selección aplicado (MAPE + extrapolación).
- [ ] Descarte de Random Forest, XGBoost y Prophet con argumento técnico.
- [ ] Clasificación de carreras por tier de confianza presentada.
- [ ] Walk-forward reportado, con explicación de los folds tempranos altos.
- [ ] Validación walk-forward de 5 períodos reportada (MAPE 5,93%, errores 10,78% → 3,32%).
- [ ] Validación out-of-sample contra 2026 P1 (5,31%, dentro del IC 95%).
- [ ] Consistencia entre walk-forward de 5 períodos (5,93%) y out-of-sample (5,31%) señalada.
- [ ] Diagnóstico final con limitaciones reconocidas.
- [ ] Notebook técnico completo ejecutable de principio a fin.
- [ ] Capítulo III del documento alineado con estos resultados.

---

*Notebook de defensa — modelo predictivo de matrícula PUCESA · Metodología PIPEDA-DATOS / CRISP-DM · 2026*